# Colab Baseline Runner (Costa)

Run Costa baseline modeling on Colab with Google Drive feature artifacts and DagsHub/MLflow tracking.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%cd /content
!if [ -d /content/PFE_Experiments/.git ]; then git -C /content/PFE_Experiments pull; else git clone https://github.com/aminetech26/PFE_Experiments.git; fi
%cd /content/PFE_Experiments

/content
Cloning into 'PFE_Experiments'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 575 (delta 111), reused 143 (delta 88), pack-reused 378 (from 1)
Receiving objects: 100% (575/575), 4.04 MiB | 13.48 MiB/s, done.
Resolving deltas: 100% (317/317), done.
/content/PFE_Experiments


In [6]:
%pip install -q uv
!uv sync

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 42.0 MB/s eta 0:00:0000:0100:01
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 316 packages in 1ms
Prepared 233 packages in 2m 00s                                          
Installed 233 packages in 1.93s                             
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0
 + alembic==1.18.4
 + amqp==5.3.1
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.12.1
 + appdirs==1.4.4
 + asyncssh==2.22.0
 + atpublic==7.0.0
 + attrs==25.4.0
 + backoff==2.2.1
 + billiard==4.2.4
 + blinker==1.9.0
 + boto3==1.42.62
 + botocore==1.42.62
 + cachetools==7.0.3
 + catboost==1.2.10
 + celery==5.6.2
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + click-didyoumean==0.3.1
 + click-plugins==1.1.1.2
 + click-repl==0.3.0
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + col

In [7]:
# Set these for your session (or use Colab secrets)
import os
os.environ['DAGSHUB_USERNAME'] = 'aminetech26'
os.environ['DAGSHUB_REPO'] = 'PFE_Experiments'
os.environ['DAGSHUB_USER_TOKEN'] = '5e31845f92874871e830dd2f59859e3d632c1aa0'

In [8]:
# Initialize Drive folders used by Optuna persistence
!uv run python -m src.training.colab_drive_init --init

Initialized tracking folders under: /content/drive/MyDrive/PV-FDD/optuna


## Drive-first feature materialization

Copy precomputed feature artifacts from Google Drive into the repo workspace.

Expected Drive source: `MyDrive/PV-FDD/data/features/costa/`

In [9]:
from pathlib import Path
import shutil

drive_features = Path('/content/drive/MyDrive/PV-FDD/data/features/costa')
repo_features = Path('/content/PFE_Experiments/data/processed/features/costa')

if not drive_features.exists():
    raise FileNotFoundError(f'Drive feature directory not found: {drive_features}')

repo_features.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(drive_features, repo_features, dirs_exist_ok=True)
print(f'Copied Costa feature artifacts to: {repo_features}')

Copied Costa feature artifacts to: /content/PFE_Experiments/data/processed/features/costa


In [10]:
print('Using precomputed Costa feature runs from Google Drive.')

Using precomputed Costa feature runs from Google Drive.


In [ ]:
# Colab's inline backend can leak into uv's venv; force a non-interactive backend for modeling runs
import os
os.environ['MPLBACKEND'] = 'Agg'

## Seed policy

Set the seed in one place directly in `configs/model_config.yaml` under `experiment.seed` before running experiments.

## Baseline Commands (per task)

Use one command per task. For each new run, update your choices in `configs/model_config.yaml` (active model, seed, HPO) and ensure the requested `task/profile/split_path` already exists in the generated feature artifacts.

In [11]:
!uv run python -m src.modeling.anomaly_detection.ml.run --task anomaly_semisup --dataset costa --split-path path_a --profile baseline_raw --run-type baseline

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/PFE_Experiments/src/modeling/anomaly_detection/ml/run.py", line 9, in <module>
    from src.modeling.anomaly_detection.ml.matrix_profile_model import run_matrix_profile
  File "/content/PFE_Experiments/src/modeling/anomaly_detection/ml/matrix_profile_model.py", line 12, in <module>
    import matplotlib.pyplot as plt
  File "/content/PFE_Experiments/.venv/lib/python3.12/site-packages/matplotlib/__init__.py", line 1299, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
    ~~~~~~~~^^^^^^^^^^^
  File "/content/PFE_Experiments/.venv/lib/python3.12/site-packages/matplotlib/__init__.py", line 774, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 